# EV scenario runner

This notebook uses the same presets as the CLI and Solara (`SCENARIOS` / `EVParams.from_scenario`) and fixed seeds for reproducibility.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from affordance_mesa.ev_params import EVParams, SCENARIOS
from affordance_mesa.ev_model import EVAdoptionModel

In [ ]:
FAST = dict(width=30, height=30, number_of_agents=60, max_steps=200)
SEED = 42
STEPS = 100

In [ ]:
results = {}

for name in sorted(SCENARIOS):
    params = EVParams.from_scenario(name, **FAST)
    model = EVAdoptionModel(params, seed=SEED)
    model.run_model(STEPS)
    results[name] = model.datacollector.get_model_vars_dataframe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name, df in results.items():
    ax.plot(df.index, df["ev_adoption_share"], label=name)

ax.set_xlabel("Step")
ax.set_ylabel("EV adoption share")
ax.set_title("EV adoption by scenario")
ax.legend()
fig.tight_layout()

In [ ]:
summary = pd.DataFrame(
    [
        {
            "scenario": name,
            "ev_adoption_share": df.iloc[-1]["ev_adoption_share"],
            "charger_count": df.iloc[-1]["charger_count"],
            "mean_adoption_score": df.iloc[-1]["mean_adoption_score"],
            "effective_ev_price": df.iloc[-1]["effective_ev_price"],
        }
        for name, df in results.items()
    ]
)
summary